In [1]:
import os
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import glob

C:\Users\jatin\AppData\Local\Temp\ipykernel_10380\1571871729.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


In [2]:
BASE_DIR = os.path.dirname(os.path.abspath(r"C:\Users\jatin\Desktop\AI Projects VS\RagApp\knowledge-base"))

KNOWLEDGE_BASE_PATH = os.path.join(BASE_DIR, "knowledge-base")

folders = glob.glob(os.path.join(KNOWLEDGE_BASE_PATH, "*"))

documents = []

for folder in folders:
    if not os.path.isdir(folder):
        continue

    # Category name: menu, business, faq
    doc_type = os.path.basename(folder)

    loader = DirectoryLoader(
        folder,
        glob="**/*.txt",
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"}
    )

    folder_docs = loader.load()

    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents")

# Display loaded file names and categories
for doc in documents:
    print(
        f"Category: {doc.metadata['doc_type']} | "
        f"Source: {doc.metadata.get('source', 'Unknown')}"
    )

Loaded 1 documents
Category: business | Source: C:\Users\jatin\Desktop\AI Projects VS\RagApp\knowledge-base\business\food-cart-info.txt


In [11]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)
print(f'Divided into {len(chunks)} chunks')

Divided into 16 chunks


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
db_name = os.path.join(BASE_DIR, "rag_vector_db")

# Load the embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

if os.path.exists(db_name):
    existing_db = Chroma(
        persist_directory=db_name,
        embedding_function=embeddings
    )
    existing_db.delete_collection()

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=db_name
)

print(
    f"Vectorstore created with "
    f"{vectorstore._collection.count()} documents"
)

C:\Users\jatin\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\jatin\AppData\Roaming\Python\Python314\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jatin\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an adminis

Vectorstore created with 16 documents


In [20]:

# Create a retriever from the vector store
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 4}
)

# Ask a question about the food cart
query = "Which burgers are available and what are their prices?"

result = retriever.invoke(query)

# Print retrieved documents
for i, doc in enumerate(result, start=1):
    print(f"\n--- Retrieved Document {i} ---")
    print(doc.page_content)
    print("Category:", doc.metadata.get("doc_type", "Unknown"))

# Combine retrieved content into context
context = "\n\n".join(
    doc.page_content for doc in result
)

print("\n--- Combined Context ---")
print(context)



--- Retrieved Document 1 ---
# Burger Menu — Food Cart

All prices are sample prices in INR. Confirm current availability before ordering.

## Classic Veg Burger
Price: ₹69
Ingredients: Vegetable patty, lettuce, onion, tomato, mayonnaise and burger bun.
Taste: Mild and savory.
Customization: No onion, no tomato, extra patty for ₹25, extra cheese for ₹20.
Allergens: Gluten; egg may be present in mayonnaise or bun depending on supplier.

## Crispy Paneer Burger
Price: ₹109
Ingredients: Crispy paneer patty, lettuce, onion, spicy mayo and bun.
Taste: Crispy, creamy and mildly spicy.
Customization: Reduce sauce, add cheese for ₹20.
Allergens: Milk, gluten; egg may be present in sauces.

## Aloo Tikki Burger
Price: ₹59
Ingredients: Potato tikki, onion, tomato, green chutney and burger bun.
Taste: Indian spices with tangy chutney.
Allergens: Gluten; check chutney ingredients.
Category: menu

--- Retrieved Document 2 ---
## Aloo Tikki Burger
Price: ₹59
Ingredients: Potato tikki, onion, tomato